In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DATA_DIR = "/kaggle/input/notebooks/abdelrahmanadel610/feature-extraction/Feature-Extraction-Output"
CSV_PATH = os.path.join(DATA_DIR, "metadata.csv")

# Ensure GPU is used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
class AudioChunkDataset(Dataset):
    def __init__(self, dataframe, data_dir):
        """
        We read the dataframe. If a file has 5 chunks, we create 5 separate entries 
        so the model can train on each chunk individually.
        """
        self.samples = []
        
        # Unroll the chunks
        for _, row in dataframe.iterrows():
            # Get the filename (e.g., '1_Normal_file.npz')
            file_name = os.path.basename(row['file_path']) 
            # Reconstruct the path based on your new kaggle input directory
            machine_folder = row['machine']
            full_path = os.path.join(data_dir, machine_folder, file_name)
            
            num_chunks = int(row['num_chunks'])
            label = int(row['label'])
            
            for chunk_idx in range(num_chunks):
                self.samples.append((full_path, chunk_idx, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, chunk_idx, label = self.samples[idx]
        
        # Load the specific chunk from the .npz file
        data = np.load(file_path)['features']
        chunk = data[chunk_idx] # Shape: (128, 94)
        
        # Convert to PyTorch Tensor
        tensor = torch.from_numpy(chunk).float()
        
        # ResNet expects images with 3 color channels (RGB).
        # Our spectrogram is 1 channel (Grayscale). So we duplicate it 3 times.
        tensor = tensor.unsqueeze(0)        # Shape becomes (1, 128, 94)
        tensor = tensor.repeat(3, 1, 1)     # Shape becomes (3, 128, 94)
        
        return tensor, label

In [3]:
# 1. Load the metadata CSV
df = pd.read_csv(CSV_PATH)

# 2. Split into Train, Val, and Test
train_val_df, test_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['label'])
train_df, val_df = train_test_split(train_val_df, test_size=0.176, random_state=42, stratify=train_val_df['label'])

# 3. CREATE THE DATASETS FIRST (This was the missing order step)
train_dataset = AudioChunkDataset(train_df, DATA_DIR)
val_dataset   = AudioChunkDataset(val_df, DATA_DIR)

# 4. NOW handle the Class Imbalance (Using the train_dataset we just made)
class_counts = train_df['label'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for _, _, label in train_dataset.samples]

# 5. Create the Sampler
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# 6. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

print(f"✅ Setup Complete!")
print(f"Train files: {len(train_df)} | Val files: {len(val_df)} | Test files: {len(test_df)}")

✅ Setup Complete!
Train files: 39387 | Val files: 8413 | Test files: 8436


In [4]:
# Load the pre-trained ResNet18 (Transfer Learning)
model = models.resnet18(weights='IMAGENET1K_V1')

# Change the final output layer to have 6 classes instead of 1000
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 6)

model = model.to(device)

# Define Loss function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 139MB/s]


In [5]:
# Calculate the total number of parameters in the model
total_params = sum(p.numel() for p in model.parameters())

# Calculate the number of TRAINABLE parameters (the ones being updated)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

if total_params < 2000000000:
    print("✅ Passed: Model is under the 2 Billion parameter limit.")
else:
    print("❌ Failed: Model is too large!")

Total Parameters: 11,179,590
Trainable Parameters: 11,179,590
✅ Passed: Model is under the 2 Billion parameter limit.


In [6]:
# 🔍 SMOKE TEST: Verify the data loading works before training
print("Running data loading test...")
try:
    # Grab the very first item from the training dataset
    test_img, test_label = train_dataset[0]
    
    print(f"✅ Success!")
    print(f"   - Image shape: {test_img.shape} (Should be [3, 128, 94])")
    print(f"   - Label:       {test_label} (Should be a number 0-5)")
    print(f"   - Values range: {test_img.min():.2f} to {test_img.max():.2f}")
    
except Exception as e:
    print(f"❌ ERROR: Could not load data. Check your file paths!")
    print(f"Details: {e}")

Running data loading test...
✅ Success!
   - Image shape: torch.Size([3, 128, 94]) (Should be [3, 128, 94])
   - Label:       3 (Should be a number 0-5)
   - Values range: -76.19 to 3.81


In [7]:
EPOCHS = 5

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in tqdm(train_loader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    train_acc = 100 * correct_train / total_train
    
    # --- VALIDATION PHASE ---
    model.eval()
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Validating"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_acc = 100 * correct_val / total_val
    
    print(f"Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

# Save the brain!
torch.save(model.state_dict(), "machine_fault_resnet18.pth")
print("✅ Model Saved Successfully!")


Epoch 1/5


Validating: 100%|██████████| 567/567 [00:51<00:00, 10.92it/s]


Train Loss: 0.0430 | Train Acc: 98.46% | Val Acc: 97.16%

Epoch 2/5


Validating: 100%|██████████| 567/567 [00:29<00:00, 19.50it/s]


Train Loss: 0.0161 | Train Acc: 99.49% | Val Acc: 99.80%

Epoch 3/5


...

Train Loss: 0.0094 | Train Acc: 99.72% | Val Acc: 98.58%

Epoch 4/5


Validating: 100%|██████████| 567/567 [00:29<00:00, 19.52it/s]


Train Loss: 0.0066 | Train Acc: 99.79% | Val Acc: 99.90%

Epoch 5/5


Validating: 100%|██████████| 567/567 [00:28<00:00, 19.78it/s]

Train Loss: 0.0056 | Train Acc: 99.81% | Val Acc: 99.15%
✅ Model Saved Successfully!


In [8]:
import collections

# Make sure model is in evaluation mode
model.eval()

correct_test_predictions = 0
total_test_files = len(test_df)

print("Starting Final Test Set Evaluation...\n")

with torch.no_grad():
    # We iterate over the test dataframe row by row (file by file)
    for _, row in tqdm(test_df.iterrows(), total=total_test_files, desc="Testing"):
        
        # 1. Load the saved .npz file
        file_name = os.path.basename(row['file_path']) 
        machine_folder = row['machine']
        full_path = os.path.join(DATA_DIR, machine_folder, file_name)
        
        true_label = int(row['label'])
        
        # Load all chunks for this specific file at once
        chunks = np.load(full_path)['features'] # Shape: (Num_Chunks, 128, 94)
        
        # 2. Convert to PyTorch Tensor and format for ResNet (3 channels)
        tensor = torch.from_numpy(chunks).float()
        tensor = tensor.unsqueeze(1)      # Shape: (Num_Chunks, 1, 128, 94)
        tensor = tensor.repeat(1, 3, 1, 1) # Shape: (Num_Chunks, 3, 128, 94)
        tensor = tensor.to(device)
        
        # 3. Get predictions for ALL chunks in this file simultaneously
        outputs = model(tensor)
        _, predicted_chunks = torch.max(outputs, 1)
        
        # 4. MAJORITY VOTING
        # Convert predictions to a python list (e.g., [3, 3, 0, 3, 3])
        predictions_list = predicted_chunks.cpu().tolist()
        
        # Find the most common prediction
        most_common_prediction = collections.Counter(predictions_list).most_common(1)[0][0]
        
        # 5. Check if the model got the whole file right
        if most_common_prediction == true_label:
            correct_test_predictions += 1

# Calculate Final Accuracy
final_accuracy = 100 * correct_test_predictions / total_test_files
print(f"\n🎉 FINAL TEST ACCURACY: {final_accuracy:.2f}%")

Starting Final Test Set Evaluation...



Testing: 100%|██████████| 8436/8436 [01:39<00:00, 85.10it/s]


🎉 FINAL TEST ACCURACY: 99.23%
